In [15]:
import pandas as pd
import time
import random
import requests
from bs4 import BeautifulSoup
import re
import json
from tqdm import tqdm
from unidecode import unidecode 
from datetime import datetime

In [16]:
def scrape_pincali():
    
    # -----------------------------
    # Consultation date
    # -----------------------------
    consultation_date = datetime.now()
    
    # -----------------------------
    # URL
    # -----------------------------
    url = "https://www.pincali.com/inmuebles/propiedades-en-renta"
    
    headers = {
        "User-Agent": "Mozilla/5.0"
    }
    
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")
    
    basic_data = soup.find_all("li", class_="property__component")
    
    # -----------------------------
    # Lists of data
    # -----------------------------
    name, type_, bedrooms, bathrooms, latitude, longitude, street, region, locality, price, surface = [], [], [], [], [], [], [], [], [], [], []
    
    # -----------------------------
    # Loop through each property and extract data
    # -----------------------------
    for elementos in basic_data:
        
        # -------- Price --------
        price_tag = elementos.find("div", class_="price")
        price.append(price_tag.get_text(strip=True) if price_tag else None)
        
        # -------- Listing --------
        name_tag = elementos.find("div", class_="title")
        name.append(name_tag.get_text(strip=True) if name_tag else None)
        
        # -------- Lat / Long --------
        lat = elementos.get("data-lat")
        lng = elementos.get("data-long")
        
        latitude.append(float(lat) if lat else None)
        longitude.append(float(lng) if lng else None)
        
        # -------- Ubicación --------
        location_tag = elementos.find("div", class_="location")
        
        if location_tag:
            location_text = location_tag.get_text(strip=True)
            street.append(location_text)
            
            partes = [p.strip() for p in location_text.split(",")]
            locality.append(partes[-1] if len(partes) > 0 else None)
            region.append(partes[-2] if len(partes) > 1 else None)
        else:
            street.append(None)
            region.append(None)
            locality.append(None)
        
        # -------- Features --------
        rec = None
        bath = None
        sup = None
        
        features = elementos.find("div", class_="features")
        
        if features:
            for item in features.find_all("div"):
                texto = item.get_text(strip=True).lower()
                
                numero = re.search(r"\d+", texto)
                numero = int(numero.group()) if numero else None
                
                if "rec" in texto:
                    rec = numero
                elif "bañ" in texto:
                    bath = numero
                elif "m²" in texto or "m2" in texto:
                    sup = numero
        
        bedrooms.append(rec)
        bathrooms.append(bath)
        surface.append(sup)
        
        type_.append(None)
    
    # -----------------------------
    # Final DataFrame
    # -----------------------------
    table = pd.DataFrame({
        "name": name,
        "price": price,
        "type": type_,
        "surface": surface,
        "bedrooms": bedrooms,
        "bathrooms": bathrooms,
        "latitude": latitude,
        "longitude": longitude,
        "street": street,
        "region": region,
        "locality": locality,
        "consultation_date": consultation_date
    })

    #Country
    country = "Mexico"
    table["country"] = country
    
    return table

In [17]:
def scrape_properati(domain: str):
    """
    Scrape Properati listings by country domain.

    Parameters
    ----------
    domain : str
        Country domain: 'ar', 'ec', 'co', 'pe'

    Returns
    -------
    pd.DataFrame
    """

    # =========================
    # Validate domain
    # =========================
    allowed_domains = ["ar", "ec", "co", "pe"]
    if domain not in allowed_domains:
        raise ValueError(f"Domain must be one of {allowed_domains}")

    # -----------------------------
    # Mapear país
    # -----------------------------
    country_map = {
        "ar": "Argentina",
        "ec": "Ecuador",
        "co": "Colombia",
        "pe": "Perú"
    }

    country = country_map[domain]

    # =========================
    # Automatic consultation date
    # =========================
    consultation_date = datetime.today().strftime("%Y-%m-%d")

    # =========================
    # URL
    # =========================
    url = f"https://www.properati.com.{domain}/s/alquiler"

    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)',
    }

    response = requests.get(url, headers=headers)

    if response.status_code != 200:
        raise ConnectionError(f"Request failed with status {response.status_code}")

    soup = BeautifulSoup(response.text, "html.parser")

    # =========================
    # BASIC INFO
    # =========================
    basics = soup.find_all("div", class_="snippet__content")

    price, surface = [], []

    for element in basics:
        price.append(
            element.find("div", class_="price").text.strip()
            if element.find("div", class_="price") else None
        )
        surface.append(
            element.find("span", class_="properties__area").text.strip()
            if element.find("span", class_="properties__area") else None
        )

    basics_df = pd.DataFrame({"price": price, "surface": surface})

    if len(basics_df) > 2:
        basics_df = basics_df.drop([0, 1]).reset_index(drop=True)

    # =========================
    # JSON-LD INFO
    # =========================
    scripts = soup.find_all("script", type="application/ld+json")

    if not scripts:
        raise ValueError("No JSON-LD data found")

    data_json = json.loads(scripts[0].text)[0]['about']

    name, type_, bedrooms, bathrooms = [], [], [], []
    latitude, longitude, street, region, locality = [], [], [], [], []

    for item in data_json:

        type_.append(item.get('@type'))
        bedrooms.append(item.get('numberOfBedrooms'))
        bathrooms.append(item.get('numberOfBathroomsTotal'))

        geo = item.get('geo', {})
        latitude.append(geo.get('latitude'))
        longitude.append(geo.get('longitude'))

        address = item.get('address', {})
        street.append(address.get('streetAddress'))
        region.append(address.get('addressRegion'))
        locality.append(address.get('addressLocality'))

        name.append(item.get('name'))

    table = pd.DataFrame({
        'name': name,
        'type': type_,
        'street': street,
        'region': region,
        'locality': locality,
        'bedrooms': bedrooms,
        'bathrooms': bathrooms,
        'latitude': latitude,
        'longitude': longitude
    })

    # =========================
    # Merge
    # =========================
    final_df = pd.concat([table, basics_df], axis=1)

    # Add automatic consultation date
    final_df["consultation_date"] = consultation_date
    final_df["country"] = country
    return final_df

In [18]:
domains = ["ar", "ec", "co", "pe"]
#Loop through domains and concatenate results
final_results = pd.DataFrame()
for domain in tqdm(domains):
    try:
        df = scrape_properati(domain)
        final_results = pd.concat([final_results, df], ignore_index=True)
    except Exception as e:
        print(f"Error scraping domain {domain}: {e}")
final_results

100%|██████████| 4/4 [00:06<00:00,  1.61s/it]


,name,type,street,region,locality,bedrooms,bathrooms,latitude,longitude,price,surface,consultation_date,country
0,Departamento en Alquiler en Pinamar,Apartment,"Calle del Pejerrey 1201-1285, Pinamar, B7167, ...",Buenos Aires Costa Atlántica,Pinamar,1.0,1.0,-37.106663,-56.863957,USD 700,55 m²,2026-02-26,Argentina
1,Departamento en Alquiler en Pinamar,Apartment,"Avenida Enrique Shaw 226-248, Pinamar, B7167, ...",Buenos Aires Costa Atlántica,Pinamar,1.0,1.0,-37.109374,-56.86635,Consultar precio,None,2026-02-26,Argentina
2,Departamento en Alquiler en Pinamar,Apartment,"del Cazón 1201-1299, Pinamar, B7167, Buenos Ai...",Buenos Aires Costa Atlántica,Pinamar,1.0,1.0,-37.10921,-56.866824,Consultar precio,None,2026-02-26,Argentina
3,Departamento en Alquiler en Pinamar,Apartment,"De las Totoras 51-99, Pinamar, B7167, Buenos A...",Buenos Aires Costa Atlántica,Pinamar,2.0,2.0,-37.110469,-56.873088,Consultar precio,11 m²,2026-02-26,Argentina
4,Departamento en Alquiler en Pinamar,Apartment,"Calle de Las Totoras 29-99, Pinamar, Provincia...",Buenos Aires Costa Atlántica,Pinamar,1.0,1.0,-37.109735,-56.872796,Consultar precio,None,2026-02-26,Argentina
...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,Apartamento en Alquiler en Santiago de Surco,Apartment,"Av. Circunvalación del Golf los Incas 148, San...",Lima,Lima,3.0,5.0,-12.0827304,-76.96818,"USD2,950",230 m²,2026-02-26,Perú
116,Almacén en Alquiler en Santa Anita,Accommodation,"Avenida 22 de Julio, Ate, Perú",Lima,Lima,NaN,NaN,-12.0372021,-76.940812,"USD47,400",None,2026-02-26,Perú
117,Local Comercial en Alquiler en Lima Cercado,Accommodation,"Av. Guillermo Dansey 500, Lima, Perú",Lima,Lima,NaN,1.0,-12.0451651,-77.0457388,"USD4,800",24 m²,2026-02-26,Perú
118,Local Comercial en Alquiler en San Juan de Lur...,Accommodation,"Av. Los Duraznos 423, Lima, Perú",Lima,Lima,NaN,3.0,-11.9814359,-77.0046887,S/.31,"1,355 m²",2026-02-26,Perú


In [19]:
pincali_mx= scrape_pincali()

In [20]:
#Pegar ambas tabla
tabla_final=pd.concat([final_results, pincali_mx], ignore_index=True)

print("Total de registros:", len(tabla_final))
tabla_final

Total de registros: 168


,name,type,street,region,locality,bedrooms,bathrooms,latitude,longitude,price,surface,consultation_date,country
0,Departamento en Alquiler en Pinamar,Apartment,"Calle del Pejerrey 1201-1285, Pinamar, B7167, ...",Buenos Aires Costa Atlántica,Pinamar,1.0,1.0,-37.106663,-56.863957,USD 700,55 m²,2026-02-26,Argentina
1,Departamento en Alquiler en Pinamar,Apartment,"Avenida Enrique Shaw 226-248, Pinamar, B7167, ...",Buenos Aires Costa Atlántica,Pinamar,1.0,1.0,-37.109374,-56.86635,Consultar precio,None,2026-02-26,Argentina
2,Departamento en Alquiler en Pinamar,Apartment,"del Cazón 1201-1299, Pinamar, B7167, Buenos Ai...",Buenos Aires Costa Atlántica,Pinamar,1.0,1.0,-37.10921,-56.866824,Consultar precio,None,2026-02-26,Argentina
3,Departamento en Alquiler en Pinamar,Apartment,"De las Totoras 51-99, Pinamar, B7167, Buenos A...",Buenos Aires Costa Atlántica,Pinamar,2.0,2.0,-37.110469,-56.873088,Consultar precio,11 m²,2026-02-26,Argentina
4,Departamento en Alquiler en Pinamar,Apartment,"Calle de Las Totoras 29-99, Pinamar, Provincia...",Buenos Aires Costa Atlántica,Pinamar,1.0,1.0,-37.109735,-56.872796,Consultar precio,None,2026-02-26,Argentina
...,...,...,...,...,...,...,...,...,...,...,...,...,...
163,Venta de depto 2 recámaras en Arthouse Mérida ...,None,"Departamento en Altabrisa, Mérida",Departamento en Altabrisa,Mérida,2.0,2.0,20.969064,-89.621683,None,119.0,2026-02-26 23:36:20.764215,Mexico
164,"OFICINAS ADMINISTRATIVAS, EN RENTA, DOS BOCAS.",None,"Oficina en Moctezuma 1ra. Sección, Paraíso",Oficina en Moctezuma 1ra. Sección,Paraíso,NaN,NaN,18.380486,-93.233304,None,668.0,2026-02-26 23:36:20.764215,Mexico
165,Venta de depto lujo 3 recámaras en Arthouse Mé...,None,"Departamento en Altabrisa, Mérida",Departamento en Altabrisa,Mérida,3.0,3.0,20.964064,-89.626683,None,174.0,2026-02-26 23:36:20.764215,Mexico
166,"DEPARTAMENTO EN RENTA SEMI AMUEBLADO, MONTES D...",None,"Departamento en Sodzil Norte, Mérida",Departamento en Sodzil Norte,Mérida,1.0,1.0,21.04028,-89.597659,None,70.0,2026-02-26 23:36:20.764215,Mexico


In [21]:
### Save to CSV
tabla_final.to_csv("rent_lac_data_example.csv", index=False)